> 每次架构的微调/升级，都是一次重新审视 Transformer 基础核心架构的机会，或者视角。

- Almost all Chinese AI companies are trying to solve the ''curse of depth'': Qwen3.5 (layer rearrangement), deepseep's mHC, Kimi's Attention Residuals, bytedance (HC, KEEL).

- https://www.youtube.com/watch?v=rNlULI-zGcw
    - https://magazine.sebastianraschka.com/p/the-big-llm-architecture-comparison

### update

- 记忆、数理与推演
    - 语文、英语、历史和政治：90%的记忆，10%的推演
    - 地理、化学、生物：70%的记忆，30%的推演
    - 数学、物理：50%的记忆，50%的数理

### 自回归

$$
p(x_{t+1}\mid C)\propto\exp(s_{\text{语法}}+s_{\text{语义}}+s_{\text{局部搭配}}+s_{\text{长程主题}}+s_{\text{异常token影响}})
$$

- “tool_search打分策略-当前默认工具被 -1.0降权，反而污染搜索结果(模型误以为搜到了默认工具只是名字**碋**了下)”
    - 这是某语言模型的输出，**碋**，假如语言模型自回归过程中意外地蹦出这个token，为什么没有导致后续输出的崩坏

### transformers

- attention vs. mlp
    - mlp 层的核心诉求就是需求隐空间的语义信息
        - token projection
    - attention: token interaction
    - 对于transformer模型来讲最大头的参数肯定是MLP
- tokens在attention做完全局交互后，通过FFN参数中存储的训练知识来更新表示。FFN实际承担着知识库的角色。Attention负责让不同位置的token交流，FFN根据交流内容从存储知识中提取相关信息增强表示。分工挺明确的，一个管交流，一个管知识更新。
    - How might LLMs store facts | Deep Learning Chapter 7: https://www.youtube.com/watch?v=9-Jl0dxWQs8
- Transformer 非线性的来源
    - attention没有对v进行非线性变换，Attention的softmax只是用来做归一化
    - 真正非线性的来源，在于 3层 ffn 中的非线性激活函数（Ffn升维能提升特征提取能力）

### induction bias

> 模型在还没见到足够数据之前，就先带着某种世界观。
- Transformer 的基础建模单位是离散 token，且在原始设计里，对任意 token 对之间都开放了直接交互的可能；它天生缺少像卷积那样强的局部性先验，也缺少对连续空间结构的硬编码偏置。
    - CNN 的归纳偏置卷积网络，天然假设：
        - 邻近像素关系更重要
        - 同一种局部模式可以在不同位置重复出现
        - 平移后语义大体不变
    - Transformer 的归纳偏置，原生 Transformer 相比之下更“通用”：
        - 不预设局部邻域一定更重要
        - 不预设二维空间结构
        - 不预设层次化的局部到整体组合方式
        - 主要依赖数据自己学出这些规律
- Transformer 并不是把每个 token 的权重真的分得一样，而是它在原生结构上缺少强局部性与连续空间先验，把输入主要视为一串离散 token，并让模型从数据中自己学习依赖关系。对于语言这很有效，但对图像、视频、3D、物理世界这类连续空间信号，单靠这种机制往往不够，还需要额外的几何与连续性偏置。

### attention

- cross attention
    - https://www.youtube.com/watch?v=aw3H-wPuRcw
    - Q 来自一个模态（这里是图像 latent/features），K/V 来自另一个模态（这里是文本 embeddings），用来把“条件信息”注入到主干特征里。
        - 文本先过 text encoder（如 CLIP/T5）得到 token embeddings
        - 图像侧是扩散过程中的 latent / feature map（U-Net 中间特征）
    - 在扩散 U-Net 的 cross-attn block 里通常是：$\mathrm{Attn}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d}}\right)V$
        - $Q = X W_Q$, 其中 $X$ 是当前层的图像 latent/features（每个空间位置/patch 一个 token）
        - $K = T W_K, \; V = T W_V$, 其中 $T$ 是文本 token embeddings
        - 每个图像位置（Q）去“查询”文本 token（K），再把对应的文本语义（V）加权融合进图像特征，从而实现“按文本引导生成”。